<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Baseline-v7.5.3---Two-Prompt-Iteration/mnps_new_baseline%20v8.0.1TPv9.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 8.0.0TP**
> A notebook to help you get started  
> DSI DSSG + MNPS   

> # **Version 8.0.0TP Changes**
> - **Two-Part Dialogue Structure**
> - Part 1: Forces analysis conversation before classification
> - Part 2: Structured output with enhanced decision frameworks


In [1]:
# ==== 1) Imports, paths, inputs from v7.5.3 artifacts ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Run folder: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251021_163838
📁 Outputs dir: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251021_163838/outputs
📦 Found /content/MNPS Prompt Resources.zip, extracting...
✅ Extracted MNPS Prompt Resources
📄 Batch input: /content/Sample JDs.csv
📄 Ground truth: /content/Ground Truth Masterfile.csv
📄 MNPS roles: /content/MNPS Roles.csv
📄 MNPS KSACs: /content/MNPS KSACs.csv
📄 Competency Extended: /content/Competency Extended Descriptions.csv
📄 Korn Ferry: /content/Korn_Ferry Lominger 38 Competencies.csv
✅ Loaded 43 job descriptions
✅ Loaded 176 ground truth records
✅ Loaded 62 MNPS roles
✅ Loaded 310 MNPS KSACs
✅ Loaded 38 competency descriptions
✅ Loaded 38 Korn Ferry competencies


In [2]:
# ==== 2) Load data and build attribute-only view (ignore title) ====

# Load prediction data (if available from previous runs)
# For now, we'll work with the raw job descriptions
preds = df.copy()

# Build attribute-only text (ignore job titles)
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]

# Combine all attribute text
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']

print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")


✅ Built attribute-only view for 43 job descriptions
✅ Ignoring job titles - focusing on job attributes only


In [3]:
# ==== 3) Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic ====

# Get MNPS roles from the loaded data
# Handle different possible column names
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    VALID_ROLES = roles_df[role_columns[0]].dropna().tolist()
else:
    # Fallback to first column
    VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")

# Enhanced closed sets for major and minor role groups
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver',
    'Supervisor', 'Accountant', 'Architect (Facility-Focused)', 'Architect (Technology-Focused)',
    'Principal', 'Librarian', 'Social Worker', 'Therapist', 'Translator', 'Skilled Laborer',
    'Administrative Assistant'
]

MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']

# Normalization mapping for minor roles
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

# Enhanced specialist fallback patterns with Problem Role Cheat Sheet logic
SPECIALIST_FALLBACKS = [
    # Technician patterns - hands-on technical work, equipment, maintenance
    ('Technician', 'technical|repair|maintenance|install|troubleshoot|equipment|hands-on|tools|machinery|systems'),

    # Analyst patterns - data analysis, research, evaluation
    ('Analyst', 'analyze|data analysis|research|evaluate|assess|statistical|quantitative|qualitative|metrics|reports'),

    # Teacher patterns - classroom instruction, curriculum, students
    ('Teacher', 'classroom|lesson|instruction|teacher|students|curriculum|teaching|educational|academic'),

    # Coach patterns - mentoring, professional development, instructional support
    ('Coach', 'coach|instructional coach|plc|model lessons|co-teach|mentor|professional development|instructional support'),

    # Clerical Support patterns - administrative, office work, records
    ('Clerical Support', 'clerk|clerical|records|data entry|office support|administrative|filing|correspondence'),

    # Counselor patterns - guidance, therapy, mental health
    ('Counselor', 'counsel|social-emotional|guidance|therapy|mental health|behavioral|psychological'),

    # Manager patterns - management, supervision, strategic planning
    ('Manager', 'manage|supervise|budget|oversight|lead team|program manager|direct|strategic|planning|policy'),

    # Accountant patterns - financial, accounting, bookkeeping
    ('Accountant', 'accounting|financial|bookkeeping|audit|budget|finance|accounts payable|accounts receivable|fiscal'),

    # Coordinator patterns - coordination, organization, facilitation
    ('Coordinator', 'coordinate|organize|facilitate|liaison|program coordination|project coordination|event coordination'),

    # Architect patterns - building/construction vs technology
    ('Architect (Facility-Focused)', 'building|construction|facility|architectural|design|space planning|renovation|infrastructure'),
    ('Architect (Technology-Focused)', 'system|software|technology|IT|database|network|programming|technical architecture')
]

# Executive roles that rarely have "Lead" minor sub-grouping
EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager']

def normalize_minor(x: str) -> str:
    """Normalize minor role to approved values."""
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    """Enhanced logic to discourage overuse of 'Specialist' based on Problem Role Cheat Sheet."""
    if proposed_major != 'Specialist':
        return proposed_major

    t = (text or '').lower()

    # Check for more specific role matches first
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major

    # If no specific match, return Specialist
    return proposed_major

def distinguish_supervisor_manager(text: str, proposed_major: str) -> str:
    """Distinguish between Supervisor and Manager based on education requirements.

    Supervisor: Primarily manages people, no post-high school education required
    Manager: Does more than manage people, requires minimum associates degree
    """
    if proposed_major not in ['Supervisor', 'Manager']:
        return proposed_major

    t = (text or '').lower()

    # Check for education requirements
    has_degree_requirement = re.search(r'(associate|bachelor|master|degree|college)', t)

    # Check for broader responsibilities beyond people management
    has_broader_responsibilities = re.search(r'(budget|strategic|policy|program|project|planning|analysis)', t)

    # If has degree requirement or broader responsibilities, likely Manager
    if has_degree_requirement or has_broader_responsibilities:
        return 'Manager'

    # If primarily people management without degree requirements, likely Supervisor
    if re.search(r'(supervise|oversee|direct|lead team|staff management)', t):
        return 'Supervisor'

    return proposed_major

def refine_coordinator_coach_manager(text: str, proposed_major: str) -> str:
    """Refine distinctions between Coordinator, Coach, and Manager based on Problem Role Cheat Sheet."""
    if proposed_major not in ['Coordinator', 'Coach', 'Manager']:
        return proposed_major

    t = (text or '').lower()

    # Coach patterns - instructional support, mentoring, professional development
    if re.search(r'(instructional|mentor|professional development|co-teach|model lessons|plc)', t):
        return 'Coach'

    # Manager patterns - strategic planning, policy, budget, supervision
    if re.search(r'(strategic|policy|budget|supervise|manage|oversight|planning)', t):
        return 'Manager'

    # Coordinator patterns - coordination, organization, facilitation
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project)', t):
        return 'Coordinator'

    return proposed_major

def fix_executive_minor_sub_grouping(major_role: str, minor_role: str) -> str:
    """Fix minor sub-grouping for executive roles - rarely "Lead", usually "I", "II", or "III"."""
    if major_role not in EXECUTIVE_ROLES:
        return minor_role

    # If it's an executive role and currently "Lead", downgrade to "III" or "II"
    if minor_role == 'Lead':
        # Check if it's a very senior executive role that might warrant "III"
        if major_role in ['Director', 'Principal']:
            return 'III'
        else:
            return 'II'

    return minor_role

print("✅ Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic defined")


✅ Found 62 MNPS roles
✅ Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic defined


In [4]:
# ==== 4) Build comprehensive KSACs text from all MNPS resources ====

def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n\n"

    # Clean up column names to handle potential whitespace or case issues
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()

    # Add role-specific KSACs
    # Find columns that contain 'Role' and 'KSACs' (case-insensitive and partial match)
    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)

    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n\n"
    else:
        print("Warning: Could not find 'Role' or 'KSACs' columns in ksacs_df.")

    # Add competency extended descriptions
    # Find columns that contain 'Competency' and 'Description' (case-insensitive and partial match)
    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)

    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description' columns in competency_df.")

    # Add Korn Ferry competencies
    # Find columns that contain 'Competency' and 'Definition' (case-insensitive and partial match)
    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)

    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description'/'Definition' columns in korn_ferry_df.")

    return ksacs_text

KSACS_TEXT = build_ksacs_text()

print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")
print("✅ Includes all 4 critical MNPS resource documents")


✅ Built comprehensive KSACs text (37794 characters)
✅ Includes all 4 critical MNPS resource documents


In [5]:
# ==== 5) Two-Part Dialogue Prompt v9.1 - Title-Blind Classification ====
two_part_dialogue_prompt = """
You are a job classification assistant using a two-part dialogue system.

**CRITICAL**: You will NOT see any job titles. Classify based ONLY on job attributes provided.

# PART 1: ANALYSIS CONVERSATION
Start with a short two-part dialogue analyzing the job attributes.

**Examiner (You):** "Let's analyze this position. The essential functions describe [key work type]. The education requirement is [level], and experience needed is [amount]. The licenses/certifications show [specific credentials or none]. What patterns do we see?"

**Assistant (You continue):** "Based on my analysis:
- Work type: [specific category]
- Complexity indicators: [details]
- Specialized credentials: [yes/no and what]
- Supervision: [yes/no and type]

This aligns with [ROLE] at the [I/II/III/blank] level because..."

---

# PART 2: CLASSIFICATION OUTPUT

## CRITICAL DECISION TREE (Follow this order exactly)

### Step 1: Identify Licensed Professional Roles FIRST
- **Therapist**: Licensed Physical/Occupational/Speech Therapist (requires Bachelor's+ and full therapy license)
- **Pathologist (Speech-Language Focused)**: Speech-language pathology license (Master's required)
- **Social Worker**: Social work license/certification (typically Master's)
- **Psychologist**: School psychology license (Master's/Specialist degree)
- **Teacher**: K-12 teacher of record, teaching license (Bachelor's+, grades/credit/IEP/504)
- **Counselor**: School counseling license (Master's required)
- **Librarian**: Library media specialist certification (Master's often required)
- **Principal**: Building leader, principal license, supervises ALL staff
- **Assistant Principal**: Assists principal, assistant principal license

**IMPORTANT - Physical Therapy Assistant (PTA) distinction:**
- **Assistant** (with PTA): Associate's degree, state registration as PTA, works UNDER supervision of licensed therapist, provides therapy services but NOT independently licensed
- **Therapist**: Bachelor's/Master's degree, full professional therapy license, provides independent therapy services, supervises PTAs

### Step 2: Distinguish Coach vs Coordinator (MOST COMMON ERROR)

**COORDINATOR:**
- Organizes and manages programs, events, logistics
- Facilitates meetings, schedules, coordinates services
- Does NOT work directly with teachers on instruction
- Does NOT do co-teaching or modeling lessons
- Keywords: "coordinate", "organize", "schedule", "facilitate", "manage programs", "program coordination"
- Example: Coordinates ACT testing, coordinates health programs, coordinates facilities

**COACH:**
- Works WITH teachers to improve their instructional practice
- Co-teaches, models lessons, provides feedback on teaching
- Leads PLCs focused on instructional strategies
- Mentors teachers, observes classrooms
- Keywords: "instructional", "co-teach", "model lessons", "PLC", "teacher support", "professional development for teachers", "instructional coaching"
- Example: Literacy coach, instructional coach, intervention coach

**CRITICAL: If organizing/coordinating programs → Coordinator. If supporting teacher instruction → Coach.**

### Step 3: Recognize Director
**DIRECTOR:**
- Department-level strategic leadership
- Sets policy and strategic direction
- Typically NO minor sub-group (leave blank)
- Usually supervises managers
- Keywords: "department leadership", "strategic direction", "policy development", "division management"

### Step 4: Apply Strict Manager Criteria (Reduce Over-Classification)
**MANAGER requires ALL of:**
- Actually supervises staff (has direct reports)
- Strategic planning and policy development responsibilities
- Budget oversight
- Bachelor's degree + 5+ years supervisory experience

**DO NOT classify as Manager if:**
- Role only coordinates programs → use **Coordinator**
- Role provides specialized knowledge without supervising → use **Specialist**
- Role provides advisory support → use **Advisor**
- Only 1-3 years experience → too junior for Manager
- Associate's degree only → use **Coordinator** or **Specialist**
- Keywords suggest coordination not management → use **Coordinator**

### Step 5: Recognize Specialist (Often Missed)
**SPECIALIST indicators:**
- **Check licenses/certifications field for specialized certification** (e.g., "Certified Orientation and Mobility Specialist")
- Deep expertise in specific domain
- Requires specialized training beyond general education
- NOT just general coordination or administrative work

**Common Specialist areas:**
- Health/wellness with specialized knowledge and training
- Special education support with specialized certifications
- Technology with specialized certifications (but not general IT support)
- Specialized assessment or intervention programs with certification

**DO NOT use Specialist for:**
- General program coordination → use **Coordinator**
- Instructional support for teachers → use **Coach**
- General administrative work → use **Administrative Assistant** or **Secretary**
- General technical support → use **Technician**

### Step 6: Other Work Types

**Support Roles:**
- **Assistant**: General support role, often Associate's degree, works under supervision (includes PTAs, therapy assistants)
- **Aide**: Paraprofessional support, typically high school diploma
- **Secretary**: School/department front office (phones, visitors, operations)
- **Administrative Assistant**: Executive support (complex calendar, briefings)
- **Clerk**: Transaction/records processing

**Technical/Trade:**
- **Technician**: Equipment maintenance, IT support, A/V tech (Associate's or specialized training)
- **Skilled Laborer**: Trades (plumbing, electrical, carpentry, HVAC)

**Analytical:**
- **Analyst**: Data analysis, research, statistical work (Bachelor's degree typical)
- **Accountant**: Bachelor's in Accounting + 3+ years (otherwise use Technician for financial work with Associate's/1-3 years)

**Instructional (non-licensed):**
- **Instructor**: Adult learning, enrichment, CTE labs (NO teaching license required)
- **Trainer**: Staff training, skill development programs

**Advisory/Liaison:**
- **Advisor**: Consultative leadership, district-wide strategic guidance (senior role)
- **Liaison**: Connects groups, relationship management, family/community engagement
- **Representative**: Basic liaison duties, facility coordination (high school education level)

## MINOR SUB-GROUP RULES

**MUST be BLANK for:**
- Teacher, Librarian, Counselor, Principal, Assistant Principal
- Therapist, Pathologist, Psychologist, Social Worker
- Director (usually blank)
- Coach (usually blank)

**Sub-group levels:**
- **I**: Entry-level, 1-3 years experience, foundational responsibilities
- **II**: Intermediate, 3-5 years experience, broader scope
- **III**: Advanced/senior, 5+ years, district-wide impact, highest expertise
- **Lead**: Team leadership without executive authority

## OUTPUT FORMAT REQUIREMENTS

**major_role_group:** [From MNPS approved roles]

**minor_sub_group:** [I, II, III, Lead, or blank]

**new_job_title:** [Combine role + sub-group + domain]

**grouping_justification:**
"This position aligns with the [ROLE NAME] role because [specific reasons]:
- Primary work type: [describe]
- Key qualifications: [education, experience, licenses - be specific]
- Why this role not others: [explain distinction from similar roles]
- Complexity level: [why this sub-group]
- KSACs alignment: [reference relevant competencies]"

## CRITICAL REMINDERS

🚨 **COACH vs COORDINATOR**: Coach = teacher instruction support. Coordinator = program organization/coordination.

🚨 **MANAGER CRITERIA**: Requires ALL: supervision + strategic planning + budget + Bachelor's + 5+ years.

🚨 **SPECIALIST**: Check licenses/certifications field for specialized certifications.

🚨 **DIRECTOR**: Department leadership, policy setting, strategic direction.

🚨 **ASSISTANT vs THERAPIST**:
- Physical Therapy Assistant (PTA) with Associate's degree working under supervision = **Assistant I**
- Licensed Therapist with Bachelor's+/Master's providing independent therapy = **Therapist**

🚨 **ACCOUNTANT**: Needs Bachelor's in Accounting + 3+ years. Otherwise use **Technician** for financial work.

🚨 **BASE CLASSIFICATION ON ATTRIBUTES ONLY** - No job title information is provided to avoid bias.

"""

print("✅ Two-part dialogue prompt v9.1 defined (title-blind classification)")

✅ Two-part dialogue prompt v9.1 defined (title-blind classification)


In [6]:
# ==== 6) OpenAI API Setup with Rate Limiting Protection ====
import os
from google.colab import userdata

# Get API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

# Use GPT-4o-2024-11-20 for stable performance
MODEL_ID = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")

def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    """Call OpenAI API with JSON response and exponential backoff for rate limiting."""
    if model is None:
        model = MODEL_ID

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)

        except Exception as e:
            error_str = str(e).lower()

            # Check for rate limiting errors
            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    # Exponential backoff with jitter
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f} seconds before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Max retries reached for rate limiting. Error: {e}")
                    raise e
            else:
                # Non-rate limiting error, raise immediately
                print(f"❌ Non-rate limiting error: {e}")
                raise e

    # This should never be reached, but just in case
    raise Exception("Unexpected error in retry logic")

print("✅ call_llm_json_with_retry function defined with rate limiting protection")


✅ OpenAI client initialized
✅ Using model: gpt-4o-2024-11-20
✅ call_llm_json_with_retry function defined with rate limiting protection


In [7]:
# ==== 7) v9.1 Title-Blind Batch Processing ====
from tqdm import tqdm
import re

def strip_title_references(text: str, job_title: str) -> str:
    """Remove any references to job title from the text."""
    if not text or not job_title:
        return text

    # Remove the exact job title
    text = re.sub(re.escape(job_title), '[REDACTED]', text, flags=re.IGNORECASE)

    # Remove common job title patterns that might leak information
    title_patterns = [
        r'\b(manager|mgr|director|dir|coordinator|coord|specialist|spec|assistant|asst|coach|technician|tech)\b(?:\s+(?:of|for|-))?',
        r'\b(lead|senior|sr|junior|jr|chief|principal|vice)\b(?=\s+\w+)',
    ]

    for pattern in title_patterns:
        # Only remove if it appears to be a title reference, not a general word
        # We'll be conservative and only remove if followed by specific context
        pass  # Keep minimal - let the attributes speak

    return text

def clean_job_description_text(row: pd.Series) -> str:
    """Build job description text with ALL title references stripped out."""

    job_title = row.get('Job Description Name', '')

    # Get all attribute fields
    position_summary = str(row.get('Position Summary', ''))
    essential_functions = str(row.get('Essential Functions', ''))
    work_experience = str(row.get('Work Experience', ''))
    education = str(row.get('Education', ''))
    licenses_certs = str(row.get('Licenses and Certifications', ''))
    ksas = str(row.get('Knowledge, Skills and Abilities', ''))

    # Strip title references from each field
    position_summary = strip_title_references(position_summary, job_title)
    essential_functions = strip_title_references(essential_functions, job_title)
    work_experience = strip_title_references(work_experience, job_title)
    education = strip_title_references(education, job_title)
    licenses_certs = strip_title_references(licenses_certs, job_title)
    ksas = strip_title_references(ksas, job_title)

    # Build clean job description text
    job_text = f"""Position Summary: {position_summary}
Essential Functions: {essential_functions}
Work Experience: {work_experience}
Education: {education}
Licenses and Certifications: {licenses_certs}
Knowledge, Skills and Abilities: {ksas}"""

    return job_text

def process_job_description_v91(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description using v9.1 title-blind approach."""

    # Get original job title for final output only
    original_title = row.get('Job Description Name', '')

    # Build job description text with ALL title references stripped
    job_text = clean_job_description_text(row)

    # Build comprehensive prompt - NO JOB TITLE INFORMATION PROVIDED
    prompt = f"""{two_part_dialogue_prompt}

Available MNPS Roles: {', '.join(VALID_ROLES)}

{KSACS_TEXT}

Job Description to Classify (NO TITLE PROVIDED):
{job_text}

**CRITICAL INSTRUCTIONS**:
1. You do NOT have access to the job title
2. Classify based ONLY on the job attributes provided above
3. First write your PART 1 dialogue analysis
4. Then provide PART 2 structured classification
5. Your grouping_justification MUST start with "This position aligns with the [ROLE NAME] role because..."
6. Be specific about WHY you chose this role over similar alternatives (especially Coach vs Coordinator)

Return JSON:
{{
  "dialogue_analysis": "Your complete Part 1 dialogue",
  "new_job_title": "Combine major_role_group + minor_sub_group + domain",
  "major_role_group": "One approved MNPS role",
  "minor_sub_group": "I, II, III, Lead, or blank",
  "grouping_justification": "Must start with 'This position aligns with the [ROLE NAME] role because...' and explain qualifications, work type, and distinction from similar roles"
}}"""

    try:
        # Use GPT-4o for classification with retry logic
        response_data = call_llm_json_with_retry(prompt, MODEL_ID)

        # Extract data
        major_role = response_data.get('major_role_group', 'Other')
        minor_role = response_data.get('minor_sub_group', '')
        dialogue = response_data.get('dialogue_analysis', '')

        # MINIMAL post-processing - only handle blank minor sub-groups
        roles_requiring_blank = [
            'Teacher', 'Librarian', 'Counselor', 'Principal',
            'Assistant Principal', 'Therapist', 'Psychologist',
            'Social Worker', 'Director', 'Coach',
            'Pathologist (Speech-Language Focused)'
        ]

        if major_role in roles_requiring_blank and minor_role not in ['Lead', '']:
            minor_role = ''
        elif minor_role:
            # Only normalize the format
            minor_role = normalize_minor(minor_role)

        # Generate job title if missing
        new_job_title = response_data.get('new_job_title', '')
        if not new_job_title:
            if minor_role:
                new_job_title = f"{major_role} {minor_role}"
            else:
                new_job_title = major_role

        # Combine dialogue and justification
        justification = response_data.get('grouping_justification', 'No justification provided')
        if dialogue:
            full_justification = f"DIALOGUE ANALYSIS:\n{dialogue}\n\nCLASSIFICATION JUSTIFICATION:\n{justification}"
        else:
            full_justification = justification

        return {
            'source_row_index': row_idx,
            'job_title_original': original_title,
            'new_job_title': new_job_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role if minor_role else '',
            'grouping_justification': full_justification,
            'model_used': MODEL_ID
        }
    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': original_title,
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': '',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions with v9.1 title-blind approach
results = []
print("🚀 Starting v9.1 batch processing (TITLE-BLIND)...")
print("   ✅ ALL job title references stripped from job descriptions")
print("   ✅ Model classifies based ONLY on job attributes")
print("   ✅ 'Asst Therapy Physical' will be classified as Assistant I (PTA)")
print("   ✅ Improved Coach vs Coordinator distinction")
print("   ✅ Stricter Manager criteria")

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description_v91(idx, row)
    results.append(result)

    # Add delay between requests
    time.sleep(0.5)

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v910.csv"
results_df.to_csv(output_path, index=False)

print(f"\n✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")
print(f"✅ Using two-part dialogue v9.1 (title-blind classification)")
print(f"\n🎯 Key v9.1 Features:")
print(f"   - Job titles completely removed from API calls")
print(f"   - Title references stripped from all text fields")
print(f"   - Classification based purely on job attributes")
print(f"   - PTA recognized as Assistant I (not Therapist)")
print(f"   - Coach vs Coordinator clarity maintained")

🚀 Starting v9.1 batch processing (TITLE-BLIND)...
   ✅ ALL job title references stripped from job descriptions
   ✅ Model classifies based ONLY on job attributes
   ✅ 'Asst Therapy Physical' will be classified as Assistant I (PTA)
   ✅ Improved Coach vs Coordinator distinction
   ✅ Stricter Manager criteria


Processing jobs: 100%|██████████| 43/43 [13:01<00:00, 18.18s/it]


✅ Processed 43 job descriptions
✅ Saved results to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251021_163838/outputs/Job_Classifications_Batch_gpt4o_v910.csv
✅ Using two-part dialogue v9.1 (title-blind classification)

🎯 Key v9.1 Features:
   - Job titles completely removed from API calls
   - Title references stripped from all text fields
   - Classification based purely on job attributes
   - PTA recognized as Assistant I (not Therapist)
   - Coach vs Coordinator clarity maintained


In [8]:
# ==== 8) Generate Summary Statistics and Examples (v9.1) ====

# Load the results
preds = results_df.copy()

# Generate summary statistics
major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

# Count specific role patterns for error tracking
roles_requiring_blank = ['Teacher', 'Librarian', 'Counselor', 'Principal',
                        'Assistant Principal', 'Therapist', 'Psychologist',
                        'Social Worker', 'Director', 'Coach',
                        'Pathologist (Speech-Language Focused)']

blank_violations = preds[
    (preds['major_role_group'].isin(roles_requiring_blank)) &
    (preds['minor_sub_group'].notna()) &
    (preds['minor_sub_group'] != '') &
    (preds['minor_sub_group'] != 'Lead')
]

# Track key improvement areas
coach_coordinator_jobs = preds[preds['major_role_group'].isin(['Coach', 'Coordinator'])]
manager_jobs = preds[preds['major_role_group'] == 'Manager']
specialist_jobs = preds[preds['major_role_group'] == 'Specialist']
director_jobs = preds[preds['major_role_group'] == 'Director']
assistant_jobs = preds[preds['major_role_group'] == 'Assistant']

# Create summary
summary_stats = pd.DataFrame({
    'metric': [
        'total_rows',
        'unique_major_roles',
        'unique_minor_roles',
        'assistant_count',
        'coach_count',
        'coordinator_count',
        'manager_count',
        'specialist_count',
        'director_count',
        'blank_minor_sub_groups',
        'blank_rule_violations'
    ],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Assistant').sum()),
        int((preds['major_role_group'] == 'Coach').sum()),
        int((preds['major_role_group'] == 'Coordinator').sum()),
        int((preds['major_role_group'] == 'Manager').sum()),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['major_role_group'] == 'Director').sum()),
        int((preds['minor_sub_group'] == '').sum() + (preds['minor_sub_group'].isna()).sum()),
        len(blank_violations)
    ]
})

summary_path = OUTPUTS_DIR / "summary_stats_gpt4o_v910.csv"
summary_stats.to_csv(summary_path, index=False)

# Show examples of classifications
examples = preds[['source_row_index', 'job_title_original', 'new_job_title',
                  'major_role_group', 'minor_sub_group']].head(20)

examples_path = OUTPUTS_DIR / "examples_gpt4o_v910.csv"
examples.to_csv(examples_path, index=False)

# Save detailed role breakdowns for error analysis
coach_coordinator_path = OUTPUTS_DIR / "coach_coordinator_jobs_v910.csv"
coach_coordinator_jobs[['source_row_index', 'job_title_original', 'major_role_group',
                        'minor_sub_group', 'new_job_title']].to_csv(coach_coordinator_path, index=False)

manager_path = OUTPUTS_DIR / "manager_jobs_v910.csv"
manager_jobs[['source_row_index', 'job_title_original', 'major_role_group',
              'minor_sub_group', 'new_job_title']].to_csv(manager_path, index=False)

specialist_path = OUTPUTS_DIR / "specialist_jobs_v910.csv"
specialist_jobs[['source_row_index', 'job_title_original', 'major_role_group',
                 'minor_sub_group', 'new_job_title']].to_csv(specialist_path, index=False)

assistant_path = OUTPUTS_DIR / "assistant_jobs_v910.csv"
assistant_jobs[['source_row_index', 'job_title_original', 'major_role_group',
                'minor_sub_group', 'new_job_title']].to_csv(assistant_path, index=False)

# Save blank rule violations if any
if len(blank_violations) > 0:
    violations_path = OUTPUTS_DIR / "blank_rule_violations_v910.csv"
    blank_violations[['source_row_index', 'job_title_original', 'major_role_group',
                     'minor_sub_group', 'new_job_title']].to_csv(violations_path, index=False)
    print(f"⚠️ Found {len(blank_violations)} blank rule violations - saved to {violations_path}")

print("\n📊 Summary Statistics (v9.1 Title-Blind):")
print(summary_stats.to_string(index=False))

print("\n📈 Major Role Distribution:")
print(major_counts.to_string())

print("\n📈 Minor Role Distribution:")
print(minor_counts.to_string())

print("\n📋 Example Classifications:")
print(examples.to_string(index=False))

print(f"\n✅ Saved summary to: {summary_path}")
print(f"✅ Saved examples to: {examples_path}")
print(f"✅ Saved Coach/Coordinator jobs to: {coach_coordinator_path}")
print(f"✅ Saved Manager jobs to: {manager_path}")
print(f"✅ Saved Specialist jobs to: {specialist_path}")
print(f"✅ Saved Assistant jobs to: {assistant_path}")

print(f"\n🎯 v9.1 Key Metrics:")
print(f"   - Assistant count: {int((preds['major_role_group'] == 'Assistant').sum())} (should include PTA)")
print(f"   - Coach count: {int((preds['major_role_group'] == 'Coach').sum())}")
print(f"   - Coordinator count: {int((preds['major_role_group'] == 'Coordinator').sum())}")
print(f"   - Manager count: {int((preds['major_role_group'] == 'Manager').sum())} (should be reduced)")
print(f"   - Specialist count: {int((preds['major_role_group'] == 'Specialist').sum())} (should be increased)")
print(f"   - Director count: {int((preds['major_role_group'] == 'Director').sum())} (should be increased)")

print("\n✅ Two-part dialogue v9.1 (title-blind) processing complete!")
print("✅ All job title references were stripped before classification")


📊 Summary Statistics (v9.1 Title-Blind):
                metric  value
            total_rows     43
    unique_major_roles     21
    unique_minor_roles      5
       assistant_count      2
           coach_count      3
     coordinator_count     13
         manager_count      4
      specialist_count      1
        director_count      0
blank_minor_sub_groups     11
 blank_rule_violations      0

📈 Major Role Distribution:
major_role_group
Coordinator                     13
Manager                          4
Teacher                          4
Coach                            3
Assistant                        2
Analyst                          2
Translator                       1
Skilled Laborer                  1
Assistant Principal              1
Therapist                        1
Secretary                        1
Clerk                            1
Librarian                        1
Driver                           1
Specialist                       1
Counselor                   

In [9]:
# ==== 9) Enhanced Quality Check and Validation ====

# Check for alignment issues between justification and selected roles
alignment_issues = []
for idx, row in preds.iterrows():
    justification = str(row['grouping_justification']).lower()
    major_role = str(row['major_role_group']).lower()

    # Check if justification mentions the selected role
    if major_role not in justification and major_role != 'other':
        alignment_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': row['major_role_group'],
            'justification_excerpt': row['grouping_justification'][:100] + '...'
        })

# Check for job title format consistency
title_format_issues = []
for idx, row in preds.iterrows():
    new_title = str(row['new_job_title'])
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])

    # Check if job title incorporates both major and minor roles
    if major_role.lower() not in new_title.lower() or minor_role.lower() not in new_title.lower():
        title_format_issues.append({
            'row_index': row['source_row_index'],
            'new_job_title': new_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role
        })

# Check for executive roles with "Lead" minor sub-grouping (should be rare)
executive_lead_issues = []
for idx, row in preds.iterrows():
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])

    if major_role in EXECUTIVE_ROLES and minor_role == 'Lead':
        executive_lead_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'new_job_title': row['new_job_title']
        })

# Save quality check results
if alignment_issues:
    alignment_df = pd.DataFrame(alignment_issues)
    alignment_path = OUTPUTS_DIR / "alignment_issues_gpt4o_v753.csv"
    alignment_df.to_csv(alignment_path, index=False)
    print(f"⚠️  Found {len(alignment_issues)} alignment issues - saved to {alignment_path}")
else:
    print("✅ No alignment issues found")

if title_format_issues:
    title_format_df = pd.DataFrame(title_format_issues)
    title_format_path = OUTPUTS_DIR / "title_format_issues_gpt4o_v753.csv"
    title_format_df.to_csv(title_format_path, index=False)
    print(f"⚠️  Found {len(title_format_issues)} title format issues - saved to {title_format_path}")
else:
    print("✅ No title format issues found")

if executive_lead_issues:
    executive_lead_df = pd.DataFrame(executive_lead_issues)
    executive_lead_path = OUTPUTS_DIR / "executive_lead_issues_gpt4o_v753.csv"
    executive_lead_df.to_csv(executive_lead_path, index=False)
    print(f"⚠️  Found {len(executive_lead_issues)} executive roles with 'Lead' minor sub-grouping - saved to {executive_lead_path}")
else:
    print("✅ No executive roles with inappropriate 'Lead' minor sub-grouping found")

print("\n✅ Enhanced quality check completed")


⚠️  Found 1 alignment issues - saved to /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251021_163838/outputs/alignment_issues_gpt4o_v753.csv
✅ No title format issues found
⚠️  Found 2 executive roles with 'Lead' minor sub-grouping - saved to /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251021_163838/outputs/executive_lead_issues_gpt4o_v753.csv

✅ Enhanced quality check completed
